# 12장. LLM이 만든 분석 코드를 검증하는 방법

이 노트북은 `book/chapters/ch12_report_generation.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 LLM이 만든 pandas, 시각화, 머신러닝 코드 초안을 그대로 믿지 않고 **현재 데이터 구조, 병합 논리, 결과 총합, 데이터 누수, 평가 지표**를 기준으로 검증하는 것입니다.

주의: 현재 강의안 파일명은 `ch12_report_generation.md`이지만, 실제 본문 내용은 LLM 코드 생성과 검증입니다.


## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 이 노트북은 5장에서 만든 `data/processed/*_clean.csv` 파일을 사용합니다.
- 전처리 파일이 없다면 먼저 `python scripts/preprocess_data.py`를 실행하세요.
- LLM 코드 검증 결과는 `reports/` 폴더에 CSV와 Markdown으로 저장합니다.
- LLM 코드는 초안이며, 실행 전과 실행 후 검증을 모두 거칩니다.


## 1. LLM 코드는 완성본이 아니라 초안이다

LLM은 분석 코드를 빠르게 만들어 주지만, 다음과 같은 실수를 할 수 있습니다.

- 실제 데이터에 없는 컬럼명을 사용함
- 잘못된 데이터셋끼리 병합함
- 병합 후 행 수나 결측치를 확인하지 않음
- 날짜나 숫자형 변환 실패를 무시함
- 머신러닝에서 정답 컬럼을 입력값에 포함함
- accuracy만 보고 분류 모델을 평가함
- 데이터에 없는 원인을 단정함

따라서 이번 장에서는 LLM 코드 초안을 검토하는 절차를 실습합니다.


## 2. 패키지와 경로 설정

노트북 실행 위치가 프로젝트 루트인지 `notebooks/` 폴더인지에 따라 경로를 자동으로 맞춥니다.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == 'notebooks':
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('프로젝트 루트:', PROJECT_ROOT)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


## 3. 전처리 데이터 불러오기

LLM 코드 검증에는 실제 데이터셋의 컬럼과 키 관계가 필요합니다. 먼저 전처리 데이터를 불러옵니다.


In [ ]:
required_files = {
    'customers': PROCESSED_DIR / 'customers_clean.csv',
    'products': PROCESSED_DIR / 'products_clean.csv',
    'orders': PROCESSED_DIR / 'orders_clean.csv',
    'order_items': PROCESSED_DIR / 'order_items_clean.csv',
}

missing_files = [path for path in required_files.values() if not path.exists()]
if missing_files:
    for path in missing_files:
        print('누락 파일:', path)
    raise FileNotFoundError('전처리 파일이 없습니다. 먼저 python scripts/preprocess_data.py 를 실행하세요.')

datasets = {name: pd.read_csv(path) for name, path in required_files.items()}
customers = datasets['customers']
products = datasets['products']
orders = datasets['orders']
order_items = datasets['order_items']

for name, df in datasets.items():
    print(name, df.shape, list(df.columns))


## 4. 실행 전 점검: 실제 데이터 구조 확인

LLM이 만든 코드를 실행하기 전에, 코드에 등장하는 데이터셋 이름과 컬럼명이 실제 데이터에 있는지 확인해야 합니다.


In [ ]:
inventory = pd.DataFrame([
    {
        'dataset': name,
        'rows': df.shape[0],
        'columns': df.shape[1],
        'column_list': ', '.join(df.columns),
        'missing_values': int(df.isna().sum().sum()),
        'duplicated_rows': int(df.duplicated().sum()),
    }
    for name, df in datasets.items()
])

inventory.to_csv(REPORT_DIR / 'ch12_dataset_inventory.csv', index=False, encoding='utf-8-sig')
inventory


In [ ]:
required_columns = {
    'customers': ['customer_id', 'gender', 'age', 'city'],
    'products': ['product_id', 'product_name', 'category', 'price'],
    'orders': ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status'],
    'order_items': ['order_id', 'product_id', 'quantity', 'unit_price'],
}

required_column_rows = []
for dataset_name, cols in required_columns.items():
    for col in cols:
        required_column_rows.append({
            'dataset': dataset_name,
            'column': col,
            'exists': col in datasets[dataset_name].columns,
        })

required_column_check = pd.DataFrame(required_column_rows)
required_column_check.to_csv(REPORT_DIR / 'ch12_required_column_check.csv', index=False, encoding='utf-8-sig')
required_column_check


## 5. 병합 키 관계 점검

LLM이 병합 코드를 제안하면 병합 기준 컬럼이 양쪽 데이터셋에 있는지, 참조 누락이 있는지 확인해야 합니다.


In [ ]:
relationship_checks = [
    {'left_dataset': 'order_items', 'right_dataset': 'products', 'key': 'product_id', 'purpose': '상품 정보와 주문 상세 연결'},
    {'left_dataset': 'order_items', 'right_dataset': 'orders', 'key': 'order_id', 'purpose': '주문 정보와 주문 상세 연결'},
    {'left_dataset': 'orders', 'right_dataset': 'customers', 'key': 'customer_id', 'purpose': '주문 정보와 고객 정보 연결'},
]

relationship_rows = []
for check in relationship_checks:
    left_df = datasets[check['left_dataset']]
    right_df = datasets[check['right_dataset']]
    key = check['key']
    left_has_key = key in left_df.columns
    right_has_key = key in right_df.columns
    invalid_count = None
    if left_has_key and right_has_key:
        invalid_count = int((~left_df[key].isin(right_df[key])).sum())
    relationship_rows.append({
        'purpose': check['purpose'],
        'left_dataset': check['left_dataset'],
        'right_dataset': check['right_dataset'],
        'key': key,
        'left_has_key': left_has_key,
        'right_has_key': right_has_key,
        'invalid_reference_count': invalid_count,
    })

relationship_check = pd.DataFrame(relationship_rows)
relationship_check.to_csv(REPORT_DIR / 'ch12_relationship_key_check.csv', index=False, encoding='utf-8-sig')
relationship_check


## 6. pandas 코드 검증 예시: 카테고리별 매출

LLM이 카테고리별 매출 코드를 만들었다고 가정합니다. 이때 병합 전후 행 수, 카테고리 누락, `line_total` 존재 여부를 함께 확인합니다.


In [ ]:
order_items_checked = order_items.copy()
if 'line_total' not in order_items_checked.columns:
    order_items_checked['line_total'] = order_items_checked['quantity'] * order_items_checked['unit_price']

before_rows = len(order_items_checked)
sales_items = order_items_checked.merge(products, on='product_id', how='left')
after_rows = len(sales_items)

category_validation = pd.DataFrame({
    'check_item': [
        '병합 전 order_items 행 수',
        '병합 후 sales_items 행 수',
        '병합 전후 행 수 동일 여부',
        'category 결측치 수',
        'line_total 숫자형 여부',
    ],
    'value': [
        before_rows,
        after_rows,
        before_rows == after_rows,
        int(sales_items['category'].isna().sum()),
        pd.api.types.is_numeric_dtype(sales_items['line_total']),
    ],
})

category_sales = (
    sales_items
    .groupby('category', as_index=False)
    .agg(total_quantity=('quantity', 'sum'), total_sales=('line_total', 'sum'))
    .sort_values('total_sales', ascending=False)
)
category_sales['sales_ratio'] = (category_sales['total_sales'] / category_sales['total_sales'].sum() * 100).round(2)

category_validation.to_csv(REPORT_DIR / 'ch12_category_sales_validation.csv', index=False, encoding='utf-8-sig')
category_sales.to_csv(REPORT_DIR / 'ch12_category_sales_validated.csv', index=False, encoding='utf-8-sig')
display(category_validation)
category_sales


## 7. pandas 코드 검증 예시: 월별 매출

월별 매출 코드는 날짜 변환, 월 생성, 시간순 정렬, 원본 총합과 집계 총합 비교가 중요합니다.


In [ ]:
before_rows = len(order_items_checked)
order_sales = order_items_checked.merge(orders, on='order_id', how='left')
after_rows = len(order_sales)

order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
order_sales['order_month'] = order_sales['order_date'].dt.to_period('M').astype(str)

monthly_sales = (
    order_sales
    .groupby('order_month', as_index=False)
    .agg(total_sales=('line_total', 'sum'), order_count=('order_id', 'nunique'))
    .sort_values('order_month')
)
monthly_sales['avg_order_value'] = (monthly_sales['total_sales'] / monthly_sales['order_count']).round(0)

monthly_validation = pd.DataFrame({
    'check_item': [
        '병합 전 order_items 행 수',
        '병합 후 order_sales 행 수',
        '병합 전후 행 수 동일 여부',
        'order_date 변환 실패 수',
        '원본 line_total 합계',
        '월별 total_sales 합계',
        '총합 차이',
    ],
    'value': [
        before_rows,
        after_rows,
        before_rows == after_rows,
        int(order_sales['order_date'].isna().sum()),
        float(order_items_checked['line_total'].sum()),
        float(monthly_sales['total_sales'].sum()),
        float(order_items_checked['line_total'].sum() - monthly_sales['total_sales'].sum()),
    ],
})

monthly_validation.to_csv(REPORT_DIR / 'ch12_monthly_sales_validation.csv', index=False, encoding='utf-8-sig')
monthly_sales.to_csv(REPORT_DIR / 'ch12_monthly_sales_validated.csv', index=False, encoding='utf-8-sig')
display(monthly_validation)
monthly_sales


## 8. 머신러닝 코드 검증: 데이터 누수 확인

LLM이 머신러닝 코드를 만들 때는 정답 컬럼이나 정답에서 파생된 컬럼이 입력값에 포함되는지 확인해야 합니다.


In [ ]:
leakage_review = pd.DataFrame({
    'case': ['주문 취소 분류', '주문 금액 회귀', '고객 구매 금액 예측', '카테고리 매출 예측'],
    'target': ['is_cancelled', 'order_total', 'customer_total_sales', 'category_total_sales'],
    'dangerous_feature': ['order_status', 'order_total 또는 line_total 합계', '집계 이후의 총 구매 금액', '이미 계산된 카테고리 총매출'],
    'why_dangerous': [
        '정답을 만들 때 사용한 컬럼이기 때문',
        '예측해야 할 값을 입력값으로 넣는 것이기 때문',
        '미래 또는 결과 정보를 미리 넣는 것이기 때문',
        '정답 그 자체 또는 정답에 가까운 정보를 넣는 것이기 때문',
    ],
    'safe_direction': [
        'order_status 제외, 결제수단/주문금액/고객특성 사용',
        'order_total 제외, 주문 전 또는 주문 구성 정보만 사용',
        '분석 시점 이전 행동 지표만 사용',
        '예측 시점 이전 상품 속성만 사용',
    ],
})

leakage_review.to_csv(REPORT_DIR / 'ch12_ml_leakage_review.csv', index=False, encoding='utf-8-sig')
leakage_review


## 9. 오류 수정 프롬프트 템플릿 만들기

오류가 발생했을 때는 오류 메시지만 붙여 넣지 말고, 목표, 현재 컬럼, 실행 코드, 오류 메시지, 원하는 수정 방향을 함께 제공해야 합니다.


In [ ]:
error_fix_prompt = '''다음 pandas 코드에서 오류가 발생했습니다.

목표:
- products와 order_items를 product_id 기준으로 병합해 카테고리별 매출을 계산하려고 합니다.

현재 데이터 컬럼:
- products: product_id, product_name, category, price
- order_items: order_id, product_id, quantity, unit_price, line_total

실행한 코드:
[여기에 코드 붙여넣기]

오류 메시지:
[여기에 오류 메시지 붙여넣기]

요청:
1. 오류 원인을 초보자도 이해할 수 있게 설명해 주세요.
2. 실제 컬럼명만 사용해 수정 코드를 제안해 주세요.
3. 병합 후 행 수와 category 결측치 확인 코드도 포함해 주세요.
4. 데이터에 없는 컬럼명은 새로 만들지 마세요.
'''

prompt_path = REPORT_DIR / 'ch12_error_fix_prompt_template.md'
prompt_path.write_text(error_fix_prompt, encoding='utf-8')
print(error_fix_prompt)


## 10. LLM 코드 리뷰 체크리스트 만들기

반복해서 사용할 수 있는 코드 리뷰 체크리스트를 만들어 저장합니다.


In [ ]:
code_review_checklist = pd.DataFrame({
    'category': ['데이터 구조', '데이터 구조', '병합', '병합', '전처리', '전처리', '머신러닝', '머신러닝', '해석', '보안'],
    'check_item': [
        '실제 데이터셋 이름을 사용했는가?',
        '실제 컬럼명만 사용했는가?',
        '병합 기준 컬럼이 양쪽 데이터에 모두 존재하는가?',
        '병합 전후 행 수를 확인했는가?',
        '날짜와 숫자형 변환 실패를 확인했는가?',
        '결측치와 중복을 무시하지 않았는가?',
        '타깃 컬럼이 명확하게 정의되었는가?',
        '데이터 누수가 없는가?',
        '결과를 원인으로 단정하지 않았는가?',
        '개인정보나 API Key가 코드와 프롬프트에 포함되지 않았는가?',
    ],
    'status': ['미확인'] * 10,
    'memo': [''] * 10,
})

code_review_checklist.to_csv(REPORT_DIR / 'ch12_llm_code_review_checklist.csv', index=False, encoding='utf-8-sig')
code_review_checklist


## 11. 검증 요약 보고서 저장하기

검증 과정 자체를 기록하면 LLM을 활용한 분석의 신뢰도를 높일 수 있습니다. 어떤 기준으로 코드를 확인했고, 어떤 위험을 수정했는지 남깁니다.


In [ ]:
validation_summary = f'''# Chapter 12 LLM 코드 생성과 검증 요약

## 1. 코드 생성 목적

LLM을 활용해 온라인 쇼핑몰 데이터의 카테고리별 매출, 월별 매출, 머신러닝 코드 초안을 만들고 검증했습니다.

## 2. 데이터셋 인벤토리

```text
{inventory.to_string(index=False)}
```

## 3. 필수 컬럼 점검

```text
{required_column_check.to_string(index=False)}
```

## 4. 키 관계 점검

```text
{relationship_check.to_string(index=False)}
```

## 5. 카테고리별 매출 코드 검증

```text
{category_validation.to_string(index=False)}
```

## 6. 월별 매출 코드 검증

```text
{monthly_validation.to_string(index=False)}
```

## 7. 머신러닝 데이터 누수 검토

```text
{leakage_review.to_string(index=False)}
```

## 8. 검토 기준

- 실제 데이터셋 이름과 컬럼명을 사용했는지 확인했습니다.
- 병합 기준과 병합 전후 행 수를 확인했습니다.
- 날짜형과 숫자형 변환 여부를 확인했습니다.
- 머신러닝 코드에서는 데이터 누수가 없는지 확인했습니다.
- 분류 모델에서는 accuracy 외 precision, recall, f1-score를 함께 확인해야 합니다.
- 보고서 해석에서는 데이터에 없는 원인을 단정하지 않도록 수정해야 합니다.

## 9. 주의할 점

LLM이 만든 코드는 초안으로만 사용해야 하며, 최종 코드는 실제 데이터 구조와 실행 결과를 기준으로 사람이 검증해야 합니다.
'''

summary_path = REPORT_DIR / 'ch12_code_validation_summary.md'
summary_path.write_text(validation_summary, encoding='utf-8')
print('검증 요약 보고서 저장 완료:', summary_path)


## 12. 소스 모듈로 전체 검증 실행하기

위에서 단계별로 실행한 검증 흐름은 `src/llm_code_validation.py`에 함수로 정리되어 있습니다. 전체 파이프라인을 한 번에 실행할 수 있습니다.


In [ ]:
from src.llm_code_validation import run_llm_code_validation

validation_result = run_llm_code_validation(
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
)

validation_result['outputs']['code_review_checklist']


## 13. 스크립트로 한 번에 실행하기

터미널에서 프로젝트 루트 기준으로 아래 명령을 실행하면 12장 LLM 코드 검증 자료가 자동으로 생성됩니다.

```bash
python scripts/run_llm_code_validation.py
```


## 14. 실습 과제

아래 과제를 직접 해결해 보세요.

1. LLM에게 카테고리별 매출 코드를 요청하고, 컬럼명과 병합 기준을 점검하세요.
2. LLM이 만든 월별 매출 코드에 총합 검증 코드가 있는지 확인하세요.
3. 주문 취소 여부 예측 코드에서 `order_status`가 feature에 들어갔는지 확인하세요.
4. 오류 메시지를 이용해 수정 프롬프트를 작성해 보세요.
5. `ch12_llm_code_review_checklist.csv`의 status와 memo를 직접 채워 보세요.
6. 검증 결과를 `ch12_code_validation_summary.md`에 정리하세요.


In [ ]:
# 과제 1. LLM이 만든 코드에서 실제 데이터에 없는 컬럼명을 찾아보세요.
# 예시: proposed_columns = ['product_id', 'category', 'revenue']
# 실제 컬럼과 비교해 revenue가 존재하는지 확인합니다.


## 15. 정리

이번 장에서는 다음 내용을 실습했습니다.

- LLM 코드는 완성본이 아니라 초안이라는 원칙
- 실행 전 데이터셋 이름과 컬럼명 확인
- 병합 키 관계 점검
- 카테고리별 매출 코드의 행 수와 결측치 검증
- 월별 매출 코드의 날짜 변환과 총합 검증
- 머신러닝 코드의 데이터 누수 검토
- 오류 메시지를 활용한 수정 프롬프트 작성
- LLM 코드 리뷰 체크리스트 작성
- 코드 검증 요약 보고서 저장
- `src/llm_code_validation.py`와 `scripts/run_llm_code_validation.py`로 반복 실행 가능한 검증 구조 만들기

다음 장에서는 외부 데이터 수집과 자동화 흐름으로 확장합니다.
